# RQ13 — Current-Only Controlled Baseline

## Research question

> How strong and stable is the current-movement-only Digit Scan baseline under the shortcut-resistant password-level evaluation fixed in RQ12?

RQ13 begins the controlled model-ablation phase.

The earlier experiments established several constraints:

- RQ5/RQ6 showed that Previous/Next context can contain shortcuts.
- RQ7 compared Absolute and within-scan normalised acoustic representations.
- RQ11 established the complete 10-candidate scan as the decision unit.
- RQ12 fixed the grouping and anti-shortcut rules.

Before changing direction handling, repeat fusion, channel choice or model capacity, one reference result is therefore frozen.

### Baseline definition

The candidate input is **Current movement only**.

There is no Previous/Next context.

The baseline uses the strongest simple representation from the RQ7 linear LOPO comparison:

- recovered **695-D Absolute acoustic feature vector**;
- logistic candidate score;
- separate wheel × direction domains;
- A and B retained as separate observations;
- complete 10-candidate within-scan ranking;
- complete password batch held out.

No password, profile, repeat ID, candidate ordinal, physical digit or identifier metadata is a model input.

### Why reuse the frozen RQ7 scores?

RQ13 is a baseline characterisation, not another representation experiment. Reusing the already frozen RQ7 Absolute LOPO scores ensures that:

- the split is identical;
- the acoustic model is identical;
- no new tuning is introduced;
- later ablations can be compared to one fixed reference.

The seven held-out passwords are used as the statistical units.

## 1. Load the frozen current-only LOPO predictions

In [ ]:
from google.colab import drive
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

drive.mount(
    "/content/drive"
)

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE
    / "Padlock_Reproduction_v1"
)

results_candidates = [
    PROJECT_ROOT / "results",
    PROJECT_ROOT
    / "Padlock_Reproduction_v1"
    / "results",
]

RESULTS_ROOT = next(
    (
        p
        for p in results_candidates
        if (
            p
            / "07_RQ7_Absolute_vs_Relative"
            / "RQ7_decision_predictions.csv"
        ).exists()
    ),
    None,
)

if RESULTS_ROOT is None:
    raise FileNotFoundError(
        "Could not locate completed RQ7 decision predictions."
    )

SOURCE_PATH = (
    RESULTS_ROOT
    / "07_RQ7_Absolute_vs_Relative"
    / "RQ7_decision_predictions.csv"
)

RESULT_DIR = (
    RESULTS_ROOT
    / "13_RQ13_Current_Only_Baseline"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

all_predictions = pd.read_csv(
    SOURCE_PATH
)

baseline = (
    all_predictions[
        all_predictions[
            "representation"
        ]
        == "Absolute"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

assert len(
    baseline
) == 840

assert baseline[
    "decision_uid"
].nunique() == 840

assert baseline[
    "heldout_batch"
].nunique() == 7

print(
    "Decisions:",
    len(
        baseline
    ),
)

print(
    "Passwords:",
    baseline[
        "heldout_batch"
    ].nunique(),
)

## 2. Baseline metrics

In [ ]:
from sklearn.metrics import f1_score


def metric_row(
    frame
):
    return {
        "n_decisions": len(
            frame
        ),
        "top1": frame[
            "top1_hit"
        ].mean(),
        "top2": frame[
            "top2_hit"
        ].mean(),
        "top3": frame[
            "top3_hit"
        ].mean(),
        "mean_true_rank": frame[
            "true_rank"
        ].mean(),
        "median_true_rank": frame[
            "true_rank"
        ].median(),
        "mrr": frame[
            "reciprocal_rank"
        ].mean(),
        "mean_margin": frame[
            "margin"
        ].mean(),
        "macro_f1": f1_score(
            frame[
                "true_digit"
            ],
            frame[
                "pred_digit"
            ],
            labels=list(
                range(10)
            ),
            average="macro",
            zero_division=0,
        ),
    }


summary_rows = [
    {
        "scope": "Overall",
        **metric_row(
            baseline
        ),
    }
]

for wheel in [
    1,
    2,
    4,
]:
    summary_rows.append({
        "scope": f"W{wheel}",
        **metric_row(
            baseline[
                baseline[
                    "wheel"
                ]
                == wheel
            ]
        ),
    })


for direction in [
    "CCW",
    "CW",
]:
    summary_rows.append({
        "scope": direction,
        **metric_row(
            baseline[
                baseline[
                    "direction"
                ]
                == direction
            ]
        ),
    })


summary = pd.DataFrame(
    summary_rows
)

overall = summary[
    summary[
        "scope"
    ]
    == "Overall"
].iloc[
    0
]


password_summary = (
    baseline.groupby(
        [
            "heldout_batch",
            "password",
        ],
        sort=False,
    )
    .apply(
        lambda g: pd.Series(
            metric_row(
                g
            )
        ),
        include_groups=False,
    )
    .reset_index()
)


wheel_direction_summary = (
    baseline.groupby(
        [
            "wheel",
            "direction",
        ],
        sort=True,
    )
    .apply(
        lambda g: pd.Series(
            metric_row(
                g
            )
        ),
        include_groups=False,
    )
    .reset_index()
)


repeat_summary = (
    baseline.groupby(
        "repeat_id",
        sort=True,
    )
    .apply(
        lambda g: pd.Series(
            metric_row(
                g
            )
        ),
        include_groups=False,
    )
    .reset_index()
)


display(
    summary.round(
        4
    )
)

display(
    password_summary.round(
        4
    )
)

## 3. Password-level inference

In [ ]:
from scipy.stats import wilcoxon

top1_password_test = wilcoxon(
    password_summary[
        "top1"
    ].to_numpy(
        dtype=float
    )
    - 0.10,
    alternative="greater",
    method="auto",
)

rank_password_test = wilcoxon(
    password_summary[
        "mean_true_rank"
    ].to_numpy(
        dtype=float
    )
    - 5.5,
    alternative="less",
    method="auto",
)


inference = pd.DataFrame([
    {
        "metric": (
            "Password-level Top-1 vs 10% random"
        ),
        "n_passwords": len(
            password_summary
        ),
        "statistic": float(
            top1_password_test.statistic
        ),
        "p_one_sided": float(
            top1_password_test.pvalue
        ),
        "observed_mean": float(
            password_summary[
                "top1"
            ].mean()
        ),
        "chance_reference": 0.10,
    },
    {
        "metric": (
            "Password-level mean true rank "
            "vs 5.5 random"
        ),
        "n_passwords": len(
            password_summary
        ),
        "statistic": float(
            rank_password_test.statistic
        ),
        "p_one_sided": float(
            rank_password_test.pvalue
        ),
        "observed_mean": float(
            password_summary[
                "mean_true_rank"
            ].mean()
        ),
        "chance_reference": 5.5,
    },
])

display(
    inference
)

## 4. Main figures

The baseline is shown at three levels:

1. stability across held-out passwords;
2. wheel × direction domain performance;
3. overall Top-k ranking relative to random chance.

The next RQ changes direction handling; RQ13 itself does not interpret small CCW/CW differences as a direction ablation.

In [ ]:
import matplotlib.pyplot as plt

DARK_BLUE = "#315B7D"
MID_BLUE = "#6F8FA8"
LIGHT_BLUE = "#AFC5D5"
MID_GREY = "#9EA5AA"
DARK_GREY = "#596168"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def save_figure(
    fig,
    stem,
):
    fig.savefig(
        RESULT_DIR
        / f"{stem}.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        RESULT_DIR
        / f"{stem}.pdf",
        bbox_inches="tight",
    )

In [ ]:
# Figure 1 — held-out-password Top-1.

fig, ax = plt.subplots(
    figsize=(
        7.4,
        4.4,
    )
)

x = np.arange(
    len(
        password_summary
    )
)

values = (
    100
    * password_summary[
        "top1"
    ].to_numpy()
)

ax.scatter(
    x,
    values,
    s=58,
    color=DARK_BLUE,
    zorder=3,
)

overall_value = (
    100
    * overall[
        "top1"
    ]
)

for (
    xi,
    value,
) in zip(
    x,
    values,
):
    if abs(
        value
        - overall_value
    ) < 4.0:
        offset = (
            0,
            -16,
        )
        va = "top"
    else:
        offset = (
            0,
            8,
        )
        va = "bottom"

    ax.annotate(
        f"{value:.1f}%",
        xy=(
            xi,
            value,
        ),
        xytext=offset,
        textcoords="offset points",
        ha="center",
        va=va,
        fontsize=8,
        color=DARK_GREY,
    )


ax.axhline(
    overall_value,
    linewidth=1.2,
    color=MID_BLUE,
)

ax.text(
    6.62,
    overall_value
    + 1.0,
    (
        f"overall "
        f"{overall_value:.1f}%"
    ),
    ha="right",
    va="bottom",
    fontsize=8,
    color=MID_BLUE,
)


ax.axhline(
    10,
    linestyle="--",
    linewidth=1,
    color=MID_GREY,
)

ax.text(
    6.62,
    11.0,
    "random 10%",
    ha="right",
    va="bottom",
    fontsize=8,
    color=DARK_GREY,
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    password_summary[
        "password"
    ].astype(
        str
    )
)

ax.set_xlabel(
    "Held-out password"
)

ax.set_ylabel(
    "Top-1 accuracy"
)

ax.set_xlim(
    -0.3,
    6.7,
)

ax.set_ylim(
    0,
    100,
)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda y, pos: f"{y:.0f}%"
    )
)

ax.grid(
    axis="y",
    alpha=0.13,
)

fig.tight_layout()

save_figure(
    fig,
    "RQ13_Fig1_password_top1",
)

plt.show()

In [ ]:
# Figure 2 — wheel × direction descriptive baseline matrix.

matrix = (
    wheel_direction_summary.pivot(
        index="wheel",
        columns="direction",
        values="top1",
    )
    .reindex(
        index=[
            1,
            2,
            4,
        ],
        columns=[
            "CCW",
            "CW",
        ],
    )
)

fig, ax = plt.subplots(
    figsize=(
        5.6,
        4.2,
    )
)

image = ax.imshow(
    100
    * matrix.to_numpy(),
    aspect="auto",
    cmap="Blues",
    vmin=60,
    vmax=92,
)

ax.set_xticks(
    np.arange(
        2
    )
)

ax.set_xticklabels(
    [
        "CCW",
        "CW",
    ]
)

ax.set_yticks(
    np.arange(
        3
    )
)

ax.set_yticklabels(
    [
        "W1",
        "W2",
        "W4",
    ]
)

ax.set_xlabel(
    "Direction"
)

ax.set_ylabel(
    "Wheel"
)

for i in range(
    3
):
    for j in range(
        2
    ):
        value = (
            100
            * matrix.iloc[
                i,
                j,
            ]
        )

        ax.text(
            j,
            i,
            f"{value:.1f}%",
            ha="center",
            va="center",
            fontsize=9,
            color=(
                "white"
                if value
                >= 77
                else "black"
            ),
        )


cbar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.88,
)

cbar.set_label(
    "Top-1 accuracy"
)

fig.tight_layout()

save_figure(
    fig,
    "RQ13_Fig2_wheel_direction_top1",
)

plt.show()

In [ ]:
# Figure 3 — overall Top-k versus random ranking.

topk = pd.DataFrame([
    {
        "metric": "Top-1",
        "observed": overall[
            "top1"
        ],
        "chance": 0.10,
    },
    {
        "metric": "Top-2",
        "observed": overall[
            "top2"
        ],
        "chance": 0.20,
    },
    {
        "metric": "Top-3",
        "observed": overall[
            "top3"
        ],
        "chance": 0.30,
    },
])

fig, ax = plt.subplots(
    figsize=(
        6.8,
        3.9,
    )
)

y = np.arange(
    len(
        topk
    )
)

for (
    yi,
    row,
) in topk.iterrows():
    ax.plot(
        [
            100
            * row[
                "chance"
            ],
            100
            * row[
                "observed"
            ],
        ],
        [
            yi,
            yi,
        ],
        color=MID_GREY,
        linewidth=1.2,
        zorder=1,
    )


ax.scatter(
    100
    * topk[
        "chance"
    ],
    y,
    s=45,
    marker="s",
    color=LIGHT_BLUE,
    label="Random ranking",
    zorder=3,
)

ax.scatter(
    100
    * topk[
        "observed"
    ],
    y,
    s=52,
    color=DARK_BLUE,
    label="Current-only baseline",
    zorder=3,
)


for (
    yi,
    row,
) in topk.iterrows():
    value = (
        100
        * row[
            "observed"
        ]
    )

    ax.annotate(
        f"{value:.1f}%",
        xy=(
            value,
            yi,
        ),
        xytext=(
            8,
            0,
        ),
        textcoords="offset points",
        va="center",
        fontsize=8,
        color=DARK_BLUE,
    )


ax.set_yticks(
    y
)

ax.set_yticklabels(
    topk[
        "metric"
    ]
)

ax.invert_yaxis()

ax.set_xlim(
    0,
    103,
)

ax.set_xlabel(
    "Decision accuracy"
)

ax.xaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda x, pos: f"{x:.0f}%"
    )
)

ax.grid(
    axis="x",
    alpha=0.13,
)

ax.legend(
    frameon=False,
    ncol=2,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        -0.13,
    ),
)

fig.subplots_adjust(
    bottom=0.20
)

save_figure(
    fig,
    "RQ13_Fig3_topk_vs_chance",
)

topk.to_csv(
    RESULT_DIR
    / "RQ13_topk_vs_chance.csv",
    index=False,
)

plt.show()

## 5. Tables and result record

In [ ]:
baseline.to_csv(
    RESULT_DIR
    / "RQ13_decision_predictions.csv",
    index=False,
)

summary.to_csv(
    RESULT_DIR
    / "RQ13_summary_metrics.csv",
    index=False,
)

password_summary.to_csv(
    RESULT_DIR
    / "RQ13_password_summary.csv",
    index=False,
)

wheel_direction_summary.to_csv(
    RESULT_DIR
    / "RQ13_wheel_direction_summary.csv",
    index=False,
)

repeat_summary.to_csv(
    RESULT_DIR
    / "RQ13_repeat_summary.csv",
    index=False,
)

inference.to_csv(
    RESULT_DIR
    / "RQ13_password_level_tests.csv",
    index=False,
)


table1 = password_summary.copy()

for (
    source_col,
    out_col,
) in [
    (
        "top1",
        "Top-1",
    ),
    (
        "top2",
        "Top-2",
    ),
    (
        "top3",
        "Top-3",
    ),
]:
    table1[
        out_col
    ] = (
        100
        * table1[
            source_col
        ]
    ).map(
        lambda x: f"{x:.1f}%"
    )


table1[
    "Mean rank"
] = table1[
    "mean_true_rank"
].map(
    lambda x: f"{x:.2f}"
)

table1 = table1[
    [
        "password",
        "n_decisions",
        "Top-1",
        "Top-2",
        "Top-3",
        "Mean rank",
    ]
].rename(
    columns={
        "password": "Password",
        "n_decisions": (
            "N decisions"
        ),
    }
)

table1.to_csv(
    RESULT_DIR
    / "RQ13_Table1_passwords.csv",
    index=False,
)

with open(
    RESULT_DIR
    / "RQ13_Table1_passwords.tex",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        table1.to_latex(
            index=False,
            escape=True,
        )
    )


table2 = wheel_direction_summary.copy()

for (
    source_col,
    out_col,
) in [
    (
        "top1",
        "Top-1",
    ),
    (
        "top2",
        "Top-2",
    ),
    (
        "top3",
        "Top-3",
    ),
]:
    table2[
        out_col
    ] = (
        100
        * table2[
            source_col
        ]
    ).map(
        lambda x: f"{x:.1f}%"
    )


table2[
    "Mean rank"
] = table2[
    "mean_true_rank"
].map(
    lambda x: f"{x:.2f}"
)

table2 = table2[
    [
        "wheel",
        "direction",
        "n_decisions",
        "Top-1",
        "Top-2",
        "Top-3",
        "Mean rank",
    ]
].rename(
    columns={
        "wheel": "Wheel",
        "direction": "Direction",
        "n_decisions": (
            "N decisions"
        ),
    }
)

table2.to_csv(
    RESULT_DIR
    / "RQ13_Table2_domains.csv",
    index=False,
)

with open(
    RESULT_DIR
    / "RQ13_Table2_domains.tex",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        table2.to_latex(
            index=False,
            escape=True,
        )
    )

display(
    table1
)

display(
    table2
)

In [ ]:
conclusion = {
    "research_question": (
        "How strong and stable is the current-movement-only Digit Scan "
        "baseline under shortcut-resistant password-level evaluation?"
    ),
    "analysis_type": (
        "Frozen current-only baseline characterisation"
    ),
    "relation_to_prior_RQs": (
        "RQ5/RQ6 removed reliance on neighbouring context; RQ7 compared "
        "feature representations; RQ11 fixed the 10-candidate ranking "
        "decision rule; RQ12 fixed the anti-shortcut grouping contract. "
        "RQ13 now freezes the strongest simple current-only baseline as "
        "the reference for subsequent controlled ablations."
    ),
    "baseline_definition": {
        "input": (
            "One current one-digit movement per candidate; no Previous/Next context."
        ),
        "feature_representation": (
            "Absolute recovered 695-D acoustic representation."
        ),
        "model_source": (
            "Frozen RQ7 leave-one-password-out logistic candidate scores."
        ),
        "domains": (
            "Separate wheel × direction models; A/B are separate observations."
        ),
        "decision_rule": (
            "Rank the ten candidates within each complete decision."
        ),
        "model_metadata_inputs": (
            "No password, profile, repeat, candidate ordinal, physical digit "
            "or identifier metadata."
        ),
    },
    "overall": {
        "n_decisions": int(
            overall[
                "n_decisions"
            ]
        ),
        "top1": float(
            overall[
                "top1"
            ]
        ),
        "top2": float(
            overall[
                "top2"
            ]
        ),
        "top3": float(
            overall[
                "top3"
            ]
        ),
        "mean_true_rank": float(
            overall[
                "mean_true_rank"
            ]
        ),
        "mrr": float(
            overall[
                "mrr"
            ]
        ),
        "macro_f1": float(
            overall[
                "macro_f1"
            ]
        ),
    },
    "password_range": {
        "top1_min": float(
            password_summary[
                "top1"
            ].min()
        ),
        "top1_max": float(
            password_summary[
                "top1"
            ].max()
        ),
    },
    "password_level_tests": {
        "top1_vs_random_p_one_sided": float(
            top1_password_test.pvalue
        ),
        "mean_rank_vs_random_p_one_sided": float(
            rank_password_test.pvalue
        ),
    },
    "interpretation": (
        f"The current-movement-only baseline is already strong under the "
        f"RQ12 password-grouped protocol: overall Top-1 is "
        f"{100 * overall['top1']:.1f}%, Top-2 "
        f"{100 * overall['top2']:.1f}% and Top-3 "
        f"{100 * overall['top3']:.1f}%, with mean true rank "
        f"{overall['mean_true_rank']:.2f}. Held-out-password Top-1 ranges "
        f"from {100 * password_summary['top1'].min():.1f}% to "
        f"{100 * password_summary['top1'].max():.1f}%. This establishes "
        f"a meaningful current-only reference without Previous/Next context "
        f"or shortcut metadata."
    ),
    "limitation": (
        "RQ13 is a baseline, not the final architecture. It reuses the "
        "empirically stronger Absolute representation from RQ7 and keeps "
        "wheel and direction separated as known physical domains. It does "
        "not yet test whether direction pooling, repeat fusion, channel "
        "choice or higher-capacity/shared-wheel modelling improves on this "
        "reference."
    ),
    "decision": (
        "Freeze this current-only LOPO ranking result as the common reference "
        "for Phase VI controlled ablations. Change one modelling factor at a "
        "time while preserving the RQ12 grouping contract."
    ),
    "next_step": (
        "The next controlled ablation tests whether CCW and CW should be "
        "modelled separately or combined."
    ),
}

with open(
    RESULT_DIR
    / "RQ13_conclusion.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        conclusion,
        f,
        indent=2,
    )


run_info = {
    "notebook": (
        "13_RQ13_Current_Only_Baseline.ipynb"
    ),
    "input": (
        "results/07_RQ7_Absolute_vs_Relative/"
        "RQ7_decision_predictions.csv"
    ),
    "used_rows": (
        "representation == Absolute"
    ),
    "data_scope": {
        "passwords": 7,
        "decisions": 840,
        "wheels": [
            1,
            2,
            4,
        ],
        "directions": [
            "CCW",
            "CW",
        ],
        "repeats": [
            "A",
            "B",
        ],
    },
    "primary_metric": (
        "Password-grouped LOPO 10-way Top-1"
    ),
    "inference_unit": (
        "Held-out password"
    ),
}


with open(
    RESULT_DIR
    / "RQ13_run_info.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        run_info,
        f,
        indent=2,
    )


print(
    json.dumps(
        conclusion,
        indent=2,
    )
)

print(
    "\nSaved outputs:"
)

for path in sorted(
    RESULT_DIR.glob(
        "RQ13_*"
    )
):
    print(
        " -",
        path.name,
    )